In [ ]:
# ✅ Capacity Consumption Analyzer (Automated)
> **Analytics**: Top workspaces by CU consumption | Copilot vs non-Copilot attribution | Peak concurrency windows | Overutilized semantic models  
> **Value**: Avoid surprise capacity overages | Support F64/F128 upsizing justification | Strong FinOps story  
> **Capacity**: `42D2BAF8-7EDD-408E-9A16-B1D1C34AF713` | **Workspace**: FabricJumpstart

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 ▸ Configuration & Dependencies
# ─────────────────────────────────────────────────────────────────────────────
import requests, json, datetime, warnings
import pandas as pd
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

# ── Capacity & workspace identifiers (auto-populated from notebook context) ──
CAPACITY_ID      = "42D2BAF8-7EDD-408E-9A16-B1D1C34AF713"
WORKSPACE_ID     = "f2adc10e-785a-4d71-91ec-ce4ee3aefef6"
WORKSPACE_NAME   = "FabricJumpstart"

# ── Alert thresholds (adjust per your SKU limits) ────────────────────────────
# F64 SKU example: 64 CU/s baseline  — change to match your actual SKU CU/s
CAPACITY_UNITS_LIMIT      = 64          # Total CU/s for your SKU
SMOOTHING_WARNING_PCT     = 70          # Warn  at 70 % of smoothed 10-min window
THROTTLING_CRITICAL_PCT   = 90          # Alert at 90 % → throttling imminent
TEAMS_WEBHOOK_URL         = ""          # Paste your Teams Incoming Webhook URL here

# ── Time window: last 24 hours in ISO-8601 ───────────────────────────────────
NOW_UTC   = datetime.datetime.utcnow()
START_UTC = NOW_UTC - datetime.timedelta(hours=24)
TIME_FROM = START_UTC.strftime("%Y-%m-%dT%H:%M:%SZ")
TIME_TO   = NOW_UTC.strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"✅ Config loaded  |  Window: {TIME_FROM}  →  {TIME_TO}")
print(f"   Capacity : {CAPACITY_ID}")
print(f"   Workspace: {WORKSPACE_NAME} ({WORKSPACE_ID})")
print(f"   Thresholds — Warning: {SMOOTHING_WARNING_PCT}%  |  Critical: {THROTTLING_CRITICAL_PCT}%")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 ▸ Authentication — acquire Power BI / Fabric bearer token
# ─────────────────────────────────────────────────────────────────────────────
try:
    # notebookutils.credentials.getToken('pbi') works for interactive (user) sessions.
    # For Service Principal pipelines, use MSAL: https://aka.ms/fabric-sp-auth
    token = notebookutils.credentials.getToken('pbi')
    HEADERS = {
        "Authorization": f"Bearer {token}",
        "Content-Type":  "application/json"
    }
    print("✅ Bearer token acquired via notebookutils (user identity)")
except Exception as e:
    print(f"⚠️  notebookutils token failed: {e}")
    print("   → For service-principal pipelines, set HEADERS manually using MSAL.")
    HEADERS = {}

BASE_URL = "https://api.fabric.microsoft.com/v1"
print(f"   Base URL: {BASE_URL}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 ▸ Pull CU usage from Fabric Monitoring REST API
#
#  Endpoint: GET /capacities/{capacityId}/operations
#  Docs    : https://learn.microsoft.com/rest/api/fabric/capacities
# ─────────────────────────────────────────────────────────────────────────────

def get_capacity_operations(capacity_id: str, time_from: str, time_to: str,
                            headers: dict, max_pages: int = 20) -> list:
    """Paginate through Fabric capacity operations for the given time window."""
    url = f"{BASE_URL}/capacities/{capacity_id}/operations"
    params = {"startDateTime": time_from, "endDateTime": time_to}
    results, page = [], 0

    while url and page < max_pages:
        resp = requests.get(url, headers=headers,
                            params=params if page == 0 else None, timeout=60)
        if resp.status_code == 401:
            raise PermissionError("401 Unauthorised — check bearer token or SP scopes.")
        if resp.status_code == 403:
            raise PermissionError("403 Forbidden — Capacity Admin role required.")
        resp.raise_for_status()
        data = resp.json()
        results.extend(data.get("value", []))
        url = data.get("continuationUri")   # next page (None → stop)
        params, page = None, page + 1

    return results


try:
    raw_ops = get_capacity_operations(CAPACITY_ID, TIME_FROM, TIME_TO, HEADERS)
    print(f"✅ Retrieved {len(raw_ops):,} operation records for the last 24 h")
    if raw_ops:
        df_raw = pd.DataFrame(raw_ops)
        print(f"   Columns: {list(df_raw.columns)}")
        display(df_raw.head(3))
    else:
        print("   ⚠️  0 records returned — verify Capacity ID and that the Monitoring API is enabled.")
        df_raw = pd.DataFrame()

except Exception as e:
    print(f"❌ API call failed: {e}")
    print("   → Generating synthetic demo data for offline validation …")
    import random, numpy as np

    workspaces = ["Finance-ETL", "Marketing-ML", "DataOps-Core",
                  "HR-Analytics", "BI-Reporting", "DevLab-Scratch"]
    item_types  = ["Spark Job", "Dataflow Gen2", "Notebook", "Pipeline", "DirectQuery Model"]
    rows = []
    for i in range(480):          # ~20 records/hour × 24 h
        ts  = START_UTC + datetime.timedelta(minutes=i * 3)
        ws  = random.choice(workspaces)
        itm = random.choice(item_types)
        cu  = round(random.expovariate(1 / 5), 2)   # right-skewed (a few big spikes)
        rows.append({
            "startTime":             ts.isoformat() + "Z",
            "workspaceId":           f"ws-{workspaces.index(ws):03d}",
            "workspaceName":         ws,
            "artifactKind":          itm,
            "artifactId":            f"art-{i % 30:04d}",
            "artifactName":          f"{ws[:4].upper()}-{itm[:4]}-{i % 10}",
            "capacityUnitSeconds":   cu,
            "status":                random.choice(["Completed"] * 3 + ["Failed"]),
        })

    df_raw = pd.DataFrame(rows)
    print(f"   Demo dataset ready: {len(df_raw):,} rows")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 ▸ Top Offenders — rank workspaces and individual items by CU spend
# ─────────────────────────────────────────────────────────────────────────────

if df_raw.empty:
    print("No data to analyse.")
else:
    # ── Normalise column names (handles real API schema vs demo schema) ────────
    col_cu   = next((c for c in df_raw.columns if "capacityUnit" in c
                     or "cuSeconds" in c.lower()), None)
    col_ws   = next((c for c in df_raw.columns if "workspaceName" in c), None) \
               or next((c for c in df_raw.columns if "workspaceId" in c), None)
    col_art  = next((c for c in df_raw.columns if "artifactName" in c
                     or "itemDisplayName" in c), None)
    col_kind = next((c for c in df_raw.columns if "artifactKind" in c
                     or "itemKind" in c), None)

    df = df_raw.copy()

    if col_cu is None:
        print("⚠️  Could not locate CU column. Available:", list(df.columns))
    else:
        df[col_cu] = pd.to_numeric(df[col_cu], errors="coerce").fillna(0)

        total_cu_24h       = df[col_cu].sum()
        CAPACITY_BUDGET_24H = CAPACITY_UNITS_LIMIT * 86_400   # CU·s in 24 h

        # ── 1. TOP WORKSPACES ─────────────────────────────────────────────────
        by_ws = (
            df.groupby(col_ws)[col_cu]
            .sum().reset_index()
            .rename(columns={col_cu: "total_cu_s", col_ws: "workspace"})
            .sort_values("total_cu_s", ascending=False)
        )
        by_ws["budget_pct"] = (by_ws["total_cu_s"] / CAPACITY_BUDGET_24H * 100).round(2)
        by_ws["flag"] = by_ws["budget_pct"].apply(
            lambda x: "🔴 CRITICAL" if x >= THROTTLING_CRITICAL_PCT
                      else ("🟡 WARNING" if x >= SMOOTHING_WARNING_PCT else "🟢 OK")
        )
        print("=" * 70)
        print(f"  TOP WORKSPACES — 24 h (budget: {CAPACITY_BUDGET_24H:,.0f} CU·s)")
        print("=" * 70)
        display(HTML(by_ws.head(10).to_html(index=False, escape=False)))

        # ── 2. TOP INDIVIDUAL ITEMS ───────────────────────────────────────────
        if col_art:
            grp = [c for c in [col_art, col_kind, col_ws] if c in df.columns]
            by_item = (
                df.groupby(grp)[col_cu]
                .agg(total_cu_s="sum", runs="count", peak_cu_s="max")
                .reset_index()
                .sort_values("total_cu_s", ascending=False)
            )
            by_item["budget_pct"] = (by_item["total_cu_s"] / CAPACITY_BUDGET_24H * 100).round(3)
            by_item["flag"] = by_item["budget_pct"].apply(
                lambda x: "🔴" if x >= 20 else ("🟡" if x >= 5 else "🟢")
            )
            print("\n")
            print("=" * 70)
            print("  TOP ITEMS — 24-HOUR CU OFFENDERS")
            print("=" * 70)
            display(HTML(by_item.head(15).to_html(index=False, escape=False)))

        # ── 3. FAILED RUNS ────────────────────────────────────────────────────
        if "status" in df.columns:
            failures = df[df["status"] == "Failed"]
            if not failures.empty:
                print(f"\n⚠️  {len(failures)} FAILED operations detected in last 24 h")
                fail_cols = [col_ws] + ([col_art] if col_art else [])
                fail_summary = (
                    failures.groupby(fail_cols)[col_cu]
                    .agg(fail_count="count", wasted_cu_s="sum")
                    .reset_index()
                    .sort_values("fail_count", ascending=False)
                )
                display(HTML(fail_summary.head(10).to_html(index=False, escape=False)))
            else:
                print("\n✅ No failed operations in the last 24 h")

        print(f"\n   Total CU·s consumed (24 h) : {total_cu_24h:,.1f}")
        print(f"   Budget utilisation         : {total_cu_24h / CAPACITY_BUDGET_24H * 100:.1f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 ▸ Smoothing & Throttling threshold evaluation
#
#  Microsoft Fabric uses a 10-minute rolling smoothing window.
#  If smoothed CU/s > 100 % of SKU limit → interactive jobs are throttled.
#  Sustained excess → scheduled jobs are also throttled.
# ─────────────────────────────────────────────────────────────────────────────

alert_messages = []   # ← collected here; consumed by CELL 6 (Teams alert)

if not df_raw.empty and col_cu:
    df_ts = df_raw.copy()
    df_ts[col_cu] = pd.to_numeric(df_ts[col_cu], errors="coerce").fillna(0)

    # ── Locate timestamp column ───────────────────────────────────────────────
    time_col = next(
        (c for c in df_ts.columns if "time" in c.lower() or "timestamp" in c.lower()), None
    )

    if time_col:
        df_ts[time_col] = pd.to_datetime(df_ts[time_col], utc=True, errors="coerce")
        df_ts = df_ts.dropna(subset=[time_col]).set_index(time_col).sort_index()

        # ── Resample to 10-min buckets (mirrors Fabric smoothing window) ──────
        smoothed = (
            df_ts[[col_cu]]
            .resample("10min").sum()
            .rename(columns={col_cu: "cu_s_per_10min"})
        )
        smoothed["cu_per_s"]         = smoothed["cu_s_per_10min"] / 600
        smoothed["utilisation_pct"]  = (smoothed["cu_per_s"] / CAPACITY_UNITS_LIMIT * 100).round(2)
        smoothed["smoothing_status"] = smoothed["utilisation_pct"].apply(
            lambda x: "🔴 THROTTLING"         if x >= THROTTLING_CRITICAL_PCT
                      else ("🟡 SMOOTHING WARN" if x >= SMOOTHING_WARNING_PCT else "🟢 OK")
        )

        peak_row = smoothed.loc[smoothed["utilisation_pct"].idxmax()]
        avg_util = smoothed["utilisation_pct"].mean()

        print("=" * 70)
        print("  ⚡ SMOOTHING WINDOW ANALYSIS  (10-min buckets, last 8 h shown)")
        print("=" * 70)
        display(HTML(smoothed.tail(48).to_html(escape=False)))   # last 8 h
        print(f"\n  Peak utilisation : {peak_row['utilisation_pct']:.1f}%  at {peak_row.name}")
        print(f"  Average (24 h)   : {avg_util:.1f}%")
        print(f"  SKU Limit        : {CAPACITY_UNITS_LIMIT} CU/s")

        # ── Evaluate breach windows and build alert list ──────────────────────
        throttle_wins  = smoothed[smoothed["utilisation_pct"] >= THROTTLING_CRITICAL_PCT]
        smoothing_wins = smoothed[
            (smoothed["utilisation_pct"] >= SMOOTHING_WARNING_PCT) &
            (smoothed["utilisation_pct"] <  THROTTLING_CRITICAL_PCT)
        ]

        if not throttle_wins.empty:
            msg = (
                f"🔴 CRITICAL — Throttling threshold ({THROTTLING_CRITICAL_PCT}%) breached "
                f"in {len(throttle_wins)} 10-min window(s). "
                f"Peak: {peak_row['utilisation_pct']:.1f}% at {peak_row.name}"
            )
            print(f"\n{msg}")
            alert_messages.append(("CRITICAL", msg))

        if not smoothing_wins.empty:
            msg = (
                f"🟡 WARNING — Smoothing warning ({SMOOTHING_WARNING_PCT}%) exceeded "
                f"in {len(smoothing_wins)} 10-min window(s). "
                f"Consider deferring non-critical Spark jobs."
            )
            print(f"\n{msg}")
            alert_messages.append(("WARNING", msg))

        if not alert_messages:
            print("\n✅ Capacity utilisation is within healthy thresholds — no alerts needed.")

    else:
        print("⚠️  No timestamp column found; skipping smoothing analysis.")
        alert_messages.append(("INFO", "Smoothing analysis skipped — no timestamp in API response."))
else:
    print("No data available for threshold analysis.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 ▸ Teams notification via Incoming Webhook (Adaptive Card)
#
#  Setup:  Teams channel → Manage channel → Connectors → Incoming Webhook
#          Paste the generated URL into TEAMS_WEBHOOK_URL in CELL 1.
#
#  Fabric Activator alternative: call POST /activator/triggers/{triggerId}
#  with the same payload to create a real-time reflex rule instead.
# ─────────────────────────────────────────────────────────────────────────────

def build_adaptive_card(alerts, workspace_name, capacity_id,
                        window_start, window_end):
    """Return a Teams Adaptive Card payload representing the health check result."""
    sev_rank = {"CRITICAL": 2, "WARNING": 1, "INFO": 0, "NONE": -1}
    overall  = max(alerts, key=lambda t: sev_rank.get(t[0], 0), default=("NONE", ""))[0]

    colour_map  = {"CRITICAL": "Attention", "WARNING": "Warning",
                   "INFO": "Accent",        "NONE": "Good"}
    label_map   = {"CRITICAL": "🔴 CRITICAL", "WARNING": "🟡 WARNING",
                   "INFO": "🔵 INFO",         "NONE": "🟢 HEALTHY"}

    body = [
        {
            "type":   "TextBlock",
            "text":   f"Fabric Capacity Health Check — {label_map[overall]}",
            "weight": "Bolder",
            "size":   "Large",
            "color":  colour_map[overall],
        },
        {
            "type": "FactSet",
            "facts": [
                {"title": "Workspace",   "value": workspace_name},
                {"title": "Capacity",    "value": capacity_id},
                {"title": "Window",      "value": f"{window_start}  →  {window_end}"},
                {"title": "Overall",     "value": label_map[overall]},
                {"title": "Alert count", "value": str(len(alerts))},
            ],
        },
    ]
    for sev, msg in alerts:
        body.append({
            "type":  "TextBlock",
            "text":  msg,
            "wrap":  True,
            "color": colour_map.get(sev, "Default"),
        })

    return {
        "type": "message",
        "attachments": [{
            "contentType": "application/vnd.microsoft.card.adaptive",
            "content": {
                "$schema": "http://adaptivecards.io/schemas/adaptive-card.json",
                "type":    "AdaptiveCard",
                "version": "1.4",
                "body":    body,
                "actions": [{
                    "type":  "Action.OpenUrl",
                    "title": "Open Fabric Monitoring Hub",
                    "url":   (f"https://app.fabric.microsoft.com/groups/"
                              f"{WORKSPACE_ID}/monitoring"),
                }],
            },
        }],
    }


# ── Decide whether to fire ────────────────────────────────────────────────────
fire_alert = any(s in ("CRITICAL", "WARNING") for s, _ in alert_messages)

if not fire_alert:
    print("✅ No thresholds breached — Teams notification skipped.")
else:
    card = build_adaptive_card(
        alerts         = alert_messages,
        workspace_name = WORKSPACE_NAME,
        capacity_id    = CAPACITY_ID,
        window_start   = TIME_FROM,
        window_end     = TIME_TO,
    )
    print("📨 Adaptive Card payload preview (first 1 500 chars):")
    print(json.dumps(card, indent=2)[:1500], "…\n")

    if TEAMS_WEBHOOK_URL:
        try:
            r = requests.post(TEAMS_WEBHOOK_URL, json=card, timeout=15)
            r.raise_for_status()
            print(f"✅ Teams alert delivered  (HTTP {r.status_code})")
        except Exception as ex:
            print(f"❌ Teams delivery failed: {ex}")
    else:
        print("⚠️  TEAMS_WEBHOOK_URL not set in CELL 1 — payload ready but not sent.")
        print("   Copy the JSON above into a Power Automate 'Post adaptive card' action.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FINAL CELL ▸ Executive Summary & Power BI Report Generation
# ─────────────────────────────────────────────────────────────────────────────

sev_rank   = {"CRITICAL": 2, "WARNING": 1, "INFO": 0, "NONE": -1}
top_sev    = max((s for s, _ in alert_messages), default="NONE",
                 key=lambda x: sev_rank.get(x, 0))

summary = {
    "run_timestamp_utc":      NOW_UTC.isoformat() + "Z",
    "capacity_id":            CAPACITY_ID,
    "workspace_name":         WORKSPACE_NAME,
    "window_hours":           24,
    "total_cu_consumed":      total_cu_24h if 'total_cu_24h' in dir() else 0,
    "budget_utilization_pct": (total_cu_24h / CAPACITY_BUDGET_24H * 100) if 'total_cu_24h' in dir() else 0,
    "alerts_fired":           len(alert_messages),
    "alert_severity":         top_sev,
    "copilot_operations":     copilot_summary[copilot_summary['is_copilot']==True]['operations'].sum() 
                              if 'copilot_summary' in dir() else 0,
    "peak_concurrent_ops":    peak_minute['concurrent_operations'] if 'peak_minute' in dir() else 0,
    "semantic_model_ops":     len(df_models) if 'df_models' in dir() else 0,
}

print("=" * 80)
print("  ✅ CAPACITY CONSUMPTION ANALYZER — EXECUTIVE SUMMARY")
print("=" * 80)
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k:<30} {v:.2f}")
    else:
        print(f"  {k:<30} {v}")
print("=" * 80)

# ══════════════════════════════════════════════════════════════════════════════
# POWER BI REPORT AUTO-GENERATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n📊 AUTO-GENERATING POWER BI DATASETS FOR EXECUTIVE REPORTING …\n")

# ── Dataset 1: Capacity consumption time series ──────────────────────────────
if 'smoothed' in dir() and not smoothed.empty:
    df_timeseries = smoothed.reset_index()
    df_timeseries["capacity_id"] = CAPACITY_ID
    df_timeseries["run_timestamp"] = NOW_UTC.isoformat() + "Z"
    print("✅ Dataset 1: Capacity time series (10-min buckets)")
    print(f"   Columns: {list(df_timeseries.columns)}")
    print(f"   Rows   : {len(df_timeseries)}")

# ── Dataset 2: Workspace consumption ranking ──────────────────────────────────
if 'by_ws' in dir() and not by_ws.empty:
    df_workspace_ranking = by_ws.copy()
    df_workspace_ranking["capacity_id"] = CAPACITY_ID
    df_workspace_ranking["run_timestamp"] = NOW_UTC.isoformat() + "Z"
    print("\n✅ Dataset 2: Workspace CU ranking")
    print(f"   Columns: {list(df_workspace_ranking.columns)}")
    print(f"   Rows   : {len(df_workspace_ranking)}")

# ── Dataset 3: Copilot vs non-Copilot attribution ────────────────────────────
if 'copilot_summary' in dir() and not copilot_summary.empty:
    df_copilot_report = copilot_summary.copy()
    df_copilot_report["capacity_id"] = CAPACITY_ID
    df_copilot_report["run_timestamp"] = NOW_UTC.isoformat() + "Z"
    print("\n✅ Dataset 3: Copilot attribution")
    print(f"   Columns: {list(df_copilot_report.columns)}")
    print(f"   Rows   : {len(df_copilot_report)}")

# ── Dataset 4: Semantic model performance ─────────────────────────────────────
if 'top_models' in dir() and not top_models.empty:
    df_semantic_models = top_models.copy()
    df_semantic_models["capacity_id"] = CAPACITY_ID
    df_semantic_models["run_timestamp"] = NOW_UTC.isoformat() + "Z"
    print("\n✅ Dataset 4: Semantic model performance")
    print(f"   Columns: {list(df_semantic_models.columns)}")
    print(f"   Rows   : {len(df_semantic_models)}")

# ── Dataset 5: Concurrency & peak windows ─────────────────────────────────────
if 'hourly_pattern' in dir() and not hourly_pattern.empty:
    df_concurrency_report = hourly_pattern.copy()
    df_concurrency_report["capacity_id"] = CAPACITY_ID
    df_concurrency_report["run_timestamp"] = NOW_UTC.isoformat() + "Z"
    print("\n✅ Dataset 5: Hourly concurrency pattern")
    print(f"   Columns: {list(df_concurrency_report.columns)}")
    print(f"   Rows   : {len(df_concurrency_report)}")

# ══════════════════════════════════════════════════════════════════════════════
# PERSIST TO LAKEHOUSE FOR POWER BI DIRECT LAKE
# ══════════════════════════════════════════════════════════════════════════════

print("\n💾 LAKEHOUSE PERSISTENCE (for Power BI Direct Lake reports) …\n")
print("   Uncomment the code below after attaching a Lakehouse to this notebook:\n")

lakehouse_code = '''
# ── Set your Lakehouse path ───────────────────────────────────────────────────
LAKEHOUSE_BASE = "Tables/capacity_analytics"

# ── Write datasets to Delta tables ────────────────────────────────────────────
if 'df_timeseries' in dir():
    spark.createDataFrame(df_timeseries).write.format("delta").mode("append") \\
        .save(f"{LAKEHOUSE_BASE}/timeseries")
    
if 'df_workspace_ranking' in dir():
    spark.createDataFrame(df_workspace_ranking).write.format("delta").mode("append") \\
        .save(f"{LAKEHOUSE_BASE}/workspace_ranking")

if 'df_copilot_report' in dir():
    spark.createDataFrame(df_copilot_report).write.format("delta").mode("append") \\
        .save(f"{LAKEHOUSE_BASE}/copilot_attribution")

if 'df_semantic_models' in dir():
    spark.createDataFrame(df_semantic_models).write.format("delta").mode("append") \\
        .save(f"{LAKEHOUSE_BASE}/semantic_models")

if 'df_concurrency_report' in dir():
    spark.createDataFrame(df_concurrency_report).write.format("delta").mode("append") \\
        .save(f"{LAKEHOUSE_BASE}/concurrency_patterns")

print("✅ All datasets written to Lakehouse Delta tables")
print("   → Ready for Power BI Direct Lake report creation")
'''

print(lakehouse_code)

# ══════════════════════════════════════════════════════════════════════════════
# NEXT STEPS FOR FINOPS EXCELLENCE
# ══════════════════════════════════════════════════════════════════════════════

print("\n📌 NEXT STEPS — FinOps Excellence Roadmap:\n")
print("  1️⃣  CONFIGURATION")
print("     • Set CAPACITY_UNITS_LIMIT to your actual SKU CU/s in Cell 1")
print("     • Set TEAMS_WEBHOOK_URL for automated alerting\n")

print("  2️⃣  AUTOMATION")
print("     • Schedule this notebook via Fabric Pipeline (hourly)")
print("     • Enable Fabric Activator for real-time CU spike detection\n")

print("  3️⃣  POWER BI REPORTING")
print("     • Attach a Lakehouse to this notebook")
print("     • Uncomment the Lakehouse persistence code above")
print("     • Create a Power BI report with Direct Lake mode:")
print("       ▸ Capacity utilization trends (line chart)")
print("       ▸ Workspace ranking (bar chart)")
print("       ▸ Copilot ROI (pie chart)")
print("       ▸ Peak concurrency heatmap (matrix)")
print("       ▸ Semantic model performance (table)\n")

print("  4️⃣  STAKEHOLDER COMMUNICATION")
print("     • Share auto-generated executive summary with Finance/CFO")
print("     • Justify F64 → F128 upsizing with peak load data")
print("     • Demonstrate Copilot value attribution\n")

print("  5️⃣  OPTIMIZATION")
print("     • Address broken shortcuts from OneLake Shortcut Auditor")
print("     • Optimize high-CU semantic models (incremental refresh)")
print("     • Separate Copilot workloads to dedicated capacity (optional)\n")

print("=" * 80)
print("  🎯 CAPACITY CONSUMPTION ANALYZER — COMPLETE")
print("=" * 80)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# NEW CELL ▸ Copilot vs Non-Copilot Usage Attribution
#
#  Splits CU consumption by Copilot-enabled vs traditional workloads.
#  Copilot artifacts include: Copilot for Power BI, Copilot for Data Factory, etc.
# ─────────────────────────────────────────────────────────────────────────────

if not df_raw.empty and col_cu:
    df_copilot = df_raw.copy()
    df_copilot[col_cu] = pd.to_numeric(df_copilot[col_cu], errors="coerce").fillna(0)
    
    # ── Identify Copilot-related operations ──────────────────────────────────
    # Heuristic: operations with "copilot" in artifact name/kind or specific operation types
    copilot_keywords = ["copilot", "ai-assisted", "semantic-kernel", "openai", "gpt"]
    
    def is_copilot_operation(row):
        """Check if an operation is Copilot-related based on metadata."""
        artifact_name = str(row.get(col_art, "")).lower() if col_art else ""
        artifact_kind = str(row.get(col_kind, "")).lower() if col_kind else ""
        operation_type = str(row.get("operationType", "")).lower()
        
        return any(kw in artifact_name or kw in artifact_kind or kw in operation_type
                   for kw in copilot_keywords)
    
    df_copilot["is_copilot"] = df_copilot.apply(is_copilot_operation, axis=1)
    
    # ── Aggregate by Copilot vs non-Copilot ──────────────────────────────────
    copilot_summary = (
        df_copilot.groupby("is_copilot")[col_cu]
        .agg(total_cu_s="sum", operations="count")
        .reset_index()
    )
    copilot_summary["category"] = copilot_summary["is_copilot"].map({
        True: "🤖 Copilot-enabled",
        False: "📊 Traditional workloads"
    })
    copilot_summary["pct_of_total"] = (
        copilot_summary["total_cu_s"] / copilot_summary["total_cu_s"].sum() * 100
    ).round(2)
    
    print("=" * 80)
    print("  COPILOT vs NON-COPILOT ATTRIBUTION (24 h)")
    print("=" * 80)
    display(HTML(copilot_summary[["category", "total_cu_s", "operations", "pct_of_total"]]
                 .to_html(index=False, escape=False)))
    
    # ── Breakdown by workspace for Copilot usage ──────────────────────────────
    if copilot_summary[copilot_summary["is_copilot"] == True]["total_cu_s"].sum() > 0:
        copilot_by_ws = (
            df_copilot[df_copilot["is_copilot"] == True]
            .groupby(col_ws)[col_cu]
            .sum()
            .reset_index()
            .rename(columns={col_cu: "copilot_cu_s", col_ws: "workspace"})
            .sort_values("copilot_cu_s", ascending=False)
        )
        print("\n📊 Top Copilot-consuming workspaces:")
        display(HTML(copilot_by_ws.head(10).to_html(index=False, escape=False)))
    else:
        print("\n✅ No Copilot operations detected in the last 24 h")
        print("   (This is expected if Copilot features are not yet enabled in your capacity)")
    
    # ── Usage justification insights ──────────────────────────────────────────
    copilot_cu = copilot_summary[copilot_summary["is_copilot"] == True]["total_cu_s"].sum()
    total_cu = total_cu_24h if 'total_cu_24h' in dir() else df_copilot[col_cu].sum()
    copilot_pct = (copilot_cu / total_cu * 100) if total_cu > 0 else 0
    
    print(f"\n💡 Copilot ROI Insight:")
    print(f"   • Copilot represents {copilot_pct:.1f}% of total CU consumption")
    print(f"   • {copilot_summary[copilot_summary['is_copilot']==True]['operations'].sum()} "
          f"AI-assisted operations vs {copilot_summary[copilot_summary['is_copilot']==False]['operations'].sum()} traditional")
    if copilot_pct > 20:
        print(f"   • 🟡 Consider separating Copilot workloads to dedicated F-SKU for cost attribution")
    
else:
    print("⚠️  No data available for Copilot attribution analysis.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# NEW CELL ▸ Peak Concurrency Windows — When is Capacity Busiest?
#
#  Identifies hourly patterns and peak concurrency windows for capacity planning.
#  Supports SKU upsizing justification (e.g., F64 → F128) based on peak load.
# ─────────────────────────────────────────────────────────────────────────────

if not df_raw.empty and col_cu:
    df_concurrency = df_raw.copy()
    df_concurrency[col_cu] = pd.to_numeric(df_concurrency[col_cu], errors="coerce").fillna(0)
    
    # ── Locate timestamp column ───────────────────────────────────────────────
    time_col = next(
        (c for c in df_concurrency.columns if "time" in c.lower() or "timestamp" in c.lower()), None
    )
    
    if time_col:
        df_concurrency[time_col] = pd.to_datetime(df_concurrency[time_col], utc=True, errors="coerce")
        df_concurrency = df_concurrency.dropna(subset=[time_col])
        
        # ── Extract hour of day for pattern analysis ──────────────────────────
        df_concurrency["hour_of_day"] = df_concurrency[time_col].dt.hour
        df_concurrency["day_of_week"] = df_concurrency[time_col].dt.day_name()
        
        # ── Concurrent operations per minute (proxy for concurrency) ─────────
        df_concurrency_ts = df_concurrency.set_index(time_col).sort_index()
        
        # Resample to 1-minute intervals and count concurrent operations
        concurrent_ops = (
            df_concurrency_ts.resample("1min").size()
            .reset_index(name="concurrent_operations")
        )
        concurrent_ops_hourly = (
            df_concurrency_ts.resample("1H")
            .agg(concurrent_ops="size", total_cu_s=(col_cu, "sum"))
            .reset_index()
        )
        concurrent_ops_hourly["hour"] = concurrent_ops_hourly[time_col].dt.hour
        
        # ── Peak concurrency identification ───────────────────────────────────
        peak_minute = concurrent_ops.loc[concurrent_ops["concurrent_operations"].idxmax()]
        peak_hour = concurrent_ops_hourly.loc[concurrent_ops_hourly["concurrent_ops"].idxmax()]
        
        print("=" * 80)
        print("  PEAK CONCURRENCY WINDOWS (24 h)")
        print("=" * 80)
        print(f"  📈 Peak minute  : {peak_minute[time_col].strftime('%Y-%m-%d %H:%M UTC')}")
        print(f"     → {peak_minute['concurrent_operations']} concurrent operations")
        print(f"\n  📈 Peak hour    : {peak_hour[time_col].strftime('%Y-%m-%d %H:00 UTC')}")
        print(f"     → {peak_hour['concurrent_ops']} total operations")
        print(f"     → {peak_hour['total_cu_s']:.1f} CU·s consumed")
        
        # ── Hourly CU consumption heatmap (hour of day) ───────────────────────
        hourly_pattern = (
            df_concurrency.groupby("hour_of_day")[col_cu]
            .agg(total_cu="sum", operations="count", avg_cu_per_op="mean")
            .reset_index()
            .sort_values("total_cu", ascending=False)
        )
        
        print("\n📊 Busiest hours of the day (by CU consumption):")
        display(HTML(hourly_pattern.head(10).to_html(index=False, escape=False)))
        
        # ── Day of week pattern ────────────────────────────────────────────────
        day_pattern = (
            df_concurrency.groupby("day_of_week")[col_cu]
            .sum()
            .reset_index()
            .rename(columns={col_cu: "total_cu_s"})
            .sort_values("total_cu_s", ascending=False)
        )
        
        print("\n📅 Consumption by day of week:")
        display(HTML(day_pattern.to_html(index=False, escape=False)))
        
        # ── SKU upsizing recommendation ────────────────────────────────────────
        avg_hourly_cu_per_s = concurrent_ops_hourly["total_cu_s"].mean() / 3600
        peak_hourly_cu_per_s = peak_hour["total_cu_s"] / 3600
        
        print(f"\n💡 Capacity Planning Insight:")
        print(f"   • Average hourly load: {avg_hourly_cu_per_s:.2f} CU/s")
        print(f"   • Peak hourly load   : {peak_hourly_cu_per_s:.2f} CU/s")
        print(f"   • Current SKU limit  : {CAPACITY_UNITS_LIMIT} CU/s")
        
        if peak_hourly_cu_per_s > CAPACITY_UNITS_LIMIT * 0.8:
            next_sku = CAPACITY_UNITS_LIMIT * 2
            print(f"   • 🔴 RECOMMENDATION: Consider upsizing to F{next_sku} SKU")
            print(f"     Peak load is {peak_hourly_cu_per_s/CAPACITY_UNITS_LIMIT*100:.0f}% of current capacity")
            print(f"     Risk of throttling during peak hours")
        elif peak_hourly_cu_per_s > CAPACITY_UNITS_LIMIT * 0.6:
            print(f"   • 🟡 Monitor closely: Peak load is {peak_hourly_cu_per_s/CAPACITY_UNITS_LIMIT*100:.0f}% of capacity")
        else:
            print(f"   • ✅ Capacity headroom is healthy ({100-peak_hourly_cu_per_s/CAPACITY_UNITS_LIMIT*100:.0f}% unused at peak)")
        
    else:
        print("⚠️  No timestamp column found; skipping concurrency analysis.")
        
else:
    print("⚠️  No data available for concurrency analysis.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# NEW CELL ▸ Overutilized Semantic Models (by Refresh & Query Volume)
#
#  Identifies semantic models that are driving high CU consumption from:
#  - Frequent refreshes
#  - High query volume (DirectQuery, Live Connection)
#  - Large dataset size + complex DAX
# ─────────────────────────────────────────────────────────────────────────────

if not df_raw.empty and col_cu:
    # ── Filter for semantic model operations ──────────────────────────────────
    semantic_model_types = ["SemanticModel", "Dataset", "DirectQuery", "LiveConnection",
                            "RefreshSemanticModel", "QuerySemanticModel"]
    
    df_models = df_raw[
        df_raw[col_kind].str.contains("|".join(semantic_model_types), case=False, na=False)
    ].copy() if col_kind in df_raw.columns else pd.DataFrame()
    
    if not df_models.empty:
        df_models[col_cu] = pd.to_numeric(df_models[col_cu], errors="coerce").fillna(0)
        
        # ── Identify operation type (refresh vs query) ────────────────────────
        df_models["operation_category"] = df_models.apply(
            lambda row: "🔄 Refresh" if "refresh" in str(row.get("operationType", "")).lower()
                                        or "refresh" in str(row.get(col_art, "")).lower()
                        else "🔍 Query",
            axis=1
        )
        
        # ── Aggregate by semantic model ───────────────────────────────────────
        model_summary = (
            df_models.groupby([col_art, col_ws, "operation_category"])[col_cu]
            .agg(total_cu_s="sum", operations="count", avg_cu_per_op="mean", max_cu="max")
            .reset_index()
            .sort_values("total_cu_s", ascending=False)
        )
        
        print("=" * 80)
        print("  OVERUTILIZED SEMANTIC MODELS (by Refresh & Query Volume)")
        print("=" * 80)
        print(f"  Total semantic model operations: {len(df_models):,}")
        print(f"  Total CU consumption: {df_models[col_cu].sum():,.1f} CU·s\n")
        
        # ── Top models by total CU consumption ────────────────────────────────
        top_models = (
            df_models.groupby([col_art, col_ws])[col_cu]
            .agg(total_cu_s="sum", refreshes="count")
            .reset_index()
            .sort_values("total_cu_s", ascending=False)
        )
        
        print("📊 Top semantic models by CU consumption:")
        display(HTML(top_models.head(15).to_html(index=False, escape=False)))
        
        # ── Breakdown by operation type ───────────────────────────────────────
        operation_breakdown = (
            df_models.groupby("operation_category")[col_cu]
            .agg(total_cu_s="sum", operations="count")
            .reset_index()
        )
        operation_breakdown["pct_of_total"] = (
            operation_breakdown["total_cu_s"] / operation_breakdown["total_cu_s"].sum() * 100
        ).round(2)
        
        print("\n📈 Refresh vs Query breakdown:")
        display(HTML(operation_breakdown.to_html(index=False, escape=False)))
        
        # ── Detect high-frequency refresh patterns ────────────────────────────
        refresh_ops = df_models[df_models["operation_category"] == "🔄 Refresh"]
        if not refresh_ops.empty:
            high_refresh_models = (
                refresh_ops.groupby([col_art, col_ws])
                .size()
                .reset_index(name="refresh_count")
                .sort_values("refresh_count", ascending=False)
            )
            high_refresh_models["refreshes_per_hour"] = (
                high_refresh_models["refresh_count"] / 24
            ).round(2)
            
            # Flag models refreshing more than once per hour
            overrefreshed = high_refresh_models[high_refresh_models["refreshes_per_hour"] > 1]
            
            if not overrefreshed.empty:
                print(f"\n⚠️  {len(overrefreshed)} model(s) refreshing > 1x/hour:")
                display(HTML(overrefreshed.to_html(index=False, escape=False)))
                print("   💡 Recommendation: Review refresh schedules; consider incremental refresh")
            else:
                print("\n✅ No excessive refresh patterns detected")
        
        # ── Identify expensive queries ─────────────────────────────────────────
        query_ops = df_models[df_models["operation_category"] == "🔍 Query"]
        if not query_ops.empty:
            query_ops_sorted = query_ops.sort_values(col_cu, ascending=False)
            expensive_queries = query_ops_sorted[query_ops_sorted[col_cu] > query_ops_sorted[col_cu].quantile(0.95)]
            
            if not expensive_queries.empty:
                print(f"\n⚠️  {len(expensive_queries)} high-cost query operation(s) detected (top 5% CU):")
                display(HTML(expensive_queries[[col_art, col_ws, col_cu, "operation_category"]]
                             .head(10).to_html(index=False, escape=False)))
                print("   💡 Recommendation: Optimize DAX; enable query caching; review DirectQuery logic")
        
        # ── FinOps recommendations ─────────────────────────────────────────────
        model_cu_pct = (df_models[col_cu].sum() / total_cu_24h * 100) if 'total_cu_24h' in dir() else 0
        print(f"\n💡 Semantic Model Optimization Opportunities:")
        print(f"   • Semantic models represent {model_cu_pct:.1f}% of total capacity consumption")
        if model_cu_pct > 40:
            print(f"   • 🔴 High semantic model usage — prioritize query & refresh optimization")
            print(f"   • Consider Premium per-user licensing for report-heavy workloads")
        elif model_cu_pct > 20:
            print(f"   • 🟡 Moderate semantic model usage — monitor refresh schedules")
        else:
            print(f"   • ✅ Semantic model usage is healthy")
        
    else:
        print("✅ No semantic model operations found in the last 24 h")
        print("   (This is expected for Spark/Notebook-heavy workloads)")
        
else:
    print("⚠️  No data available for semantic model analysis.")